# 02 - Vector Extraction

Extract a representation vector for each cultural concept and find the layer
where the concept is most linearly separable.

## The contrastive mean-difference recipe

A concept vector is a single direction in the residual stream that encodes a
cultural concept. The naive approach — averaging the activations of prompts
that express the concept — does not work: the resulting vector is dominated by
components that encode "this is an ordinary sentence in Arabic", which every
prompt shares. Every concept would end up pointing in roughly the same
direction.

The contrastive recipe removes that shared component by subtracting a baseline:

$$
v_{\text{concept}} = \frac{
  \overline{h^{(\ell)}_{\text{positive}}} - \overline{h^{(\ell)}_{\text{neutral}}}
}{
  \left\lVert \overline{h^{(\ell)}_{\text{positive}}} - \overline{h^{(\ell)}_{\text{neutral}}} \right\rVert_2
}
$$

Step by step, as implemented in `CulturalRepE.extract_vector`:

1. **Positive mean.** Run the concept's example sentences through the model and
   read `blocks.{layer}.hook_resid_post`. Average over real (non-padding)
   tokens within each prompt, then over prompts, so every prompt counts equally
   regardless of length.
2. **Negative mean.** Do the same for a set of culturally neutral sentences.
   When the dataset carries no curated negatives, `src.data.contrastive`
   supplies a deterministic bilingual bank ("The weather is nice today.",
   "الطقس جميل اليوم.", ...).
3. **Difference.** Subtract the two means. What survives is the direction that
   distinguishes concept prompts from ordinary ones.
4. **Normalise.** Scale to unit L2 norm, so that in Phase 3 the injection
   strength is the only magnitude knob.

Two implementation details matter in practice:

- **Padding is masked out.** Batching prompts of different lengths pads the
  short ones; counting those pad positions would pull short prompts toward the
  pad embedding. Only leading/trailing pads are masked, since GPT-2 style
  tokenizers reuse one id for pad, bos and eos.
- **Averaging happens in float32** even when the model runs in bfloat16.
  Summing hundreds of low-precision activations loses more signal than the cast
  costs.

### Which layer?

Extraction defaults to the **middle layer** (`n_layers // 2`). Early layers are
still close to token identity, and the last layers are specialised for
next-token prediction; mid-stack is where semantic features tend to be most
linearly separable. The sweep at the end of this notebook is how you check that
for a specific model rather than taking it on faith.

## Setup

This notebook runs end to end on CPU with a tiny randomly-initialised model, so
it can be executed anywhere. Swap `USE_TINY_MODEL = False` to run the real
extraction against the configured base model (a GPU is strongly recommended).

Configuration comes from `.env` — never hard-code a token in a notebook.

In [ ]:
import os
from pathlib import Path

import torch
from dotenv import load_dotenv

from src.data.contrastive import build_neutral_examples
from src.data.dataset_builder import load_concepts
from src.models.rep_engine import CulturalRepE

load_dotenv()

USE_TINY_MODEL = True  # set to False to use BASE_MODEL_NAME from .env

DATASET_PATH = Path("../data/datasets/cultural_concepts.jsonl")
concepts = load_concepts(DATASET_PATH)
[(c.concept_id, c.concept_en) for c in concepts]

## Build the engine

`USE_TINY_MODEL` builds a 4-layer, 16-dimensional transformer from a config and
borrows the `sshleifer/tiny-gpt2` tokenizer. No pretrained weights are
downloaded, so the numbers below are meaningless as research output — the point
is that the pipeline runs, with the exact code path used at full scale.

In [ ]:
if USE_TINY_MODEL:
    from transformer_lens import HookedTransformer, HookedTransformerConfig

    tiny_cfg = HookedTransformerConfig(
        n_layers=4,
        d_model=16,
        n_ctx=64,
        d_head=4,
        n_heads=4,
        d_mlp=32,
        act_fn="gelu",
        d_vocab=50257,
        tokenizer_name="sshleifer/tiny-gpt2",
        device="cpu",
    )
    engine = CulturalRepE(
        model_name="tiny-gpt2-random",
        device="cpu",
        dtype="float32",
        dataset_path=DATASET_PATH,
    )
    engine.model = HookedTransformer(tiny_cfg)
    engine.tokenizer = engine.model.tokenizer
else:
    engine = CulturalRepE(
        model_name=os.environ.get("BASE_MODEL_NAME", "meta-llama/Meta-Llama-3-8B-Instruct"),
        device=os.environ.get("DEVICE", "cuda"),
        dtype=os.environ.get("DTYPE", "bfloat16"),
        hf_token=os.environ.get("HF_TOKEN") or None,  # read from .env, never inline
        dataset_path=DATASET_PATH,
    )
    engine.load_model()

engine.n_layers, engine.middle_layer

## Inspect the two sides of the contrast

The positive side comes from the dataset; the negative side is generated when
the dataset has no curated negatives.

In [ ]:
concept = concepts[-1]  # diyafa_001
positives = concept.all_examples
negatives = build_neutral_examples(len(positives))

print("concept :", concept.concept_ar, "/", concept.concept_en)
print("positive:", positives)
print("negative:", negatives)

## Extract one vector

`extract_vector` accepts a `concept_id` and loads the examples from the dataset
itself, so the call is usually a one-liner.

In [ ]:
vector = engine.extract_vector(concept.concept_id, layer=engine.middle_layer)

print("shape    :", tuple(vector.shape))
print("L2 norm  :", torch.linalg.vector_norm(vector).item())
print("layer    :", engine.extraction_layers[concept.concept_id])

### Verify the recipe by hand

The same result, computed step by step, to confirm the vector really is the
normalised difference of the two means.

In [ ]:
positive_mean = engine._compute_mean_activation(positives, engine.middle_layer)
negative_mean = engine._compute_mean_activation(negatives, engine.middle_layer)

difference = positive_mean - negative_mean
manual = difference / torch.linalg.vector_norm(difference)

print("matches extract_vector:", torch.allclose(manual, vector, atol=1e-6))
print("raw difference norm   :", torch.linalg.vector_norm(difference).item())

## Extract every concept

`extract_all_vectors` iterates the dataset and skips concepts without examples
rather than aborting the run.

In [ ]:
vectors = engine.extract_all_vectors(layer=engine.middle_layer)
{name: tuple(v.shape) for name, v in vectors.items()}

## How similar are the concept directions?

Distinct concepts should not collapse onto one direction. With random weights
this plot is noise; with a real model it is the first sanity check worth
running — near-identical vectors mean the contrast is not doing its job.

In [ ]:
import numpy as np

from src.utils.visualization import plot_concept_similarity

names = list(vectors)
matrix = torch.stack([vectors[name] for name in names])
matrix = matrix / torch.linalg.vector_norm(matrix, dim=-1, keepdim=True)
similarity = (matrix @ matrix.T).numpy().astype(np.float64)

plot_concept_similarity(similarity, names)

## Layer sweep

Extract the same concept at every layer and measure how far each direction sits
from the others. With a trained model, replace this placeholder score with the
accuracy of a linear probe trained on held-out prompts — that is the measure
that tells you where the concept is actually separable.

In [ ]:
from src.utils.visualization import plot_layer_scores

scores = {}
for layer in range(engine.n_layers):
    layer_vector = engine.extract_vector(concept.concept_id, layer=layer)
    scores[layer] = torch.linalg.vector_norm(layer_vector - vector).item()

plot_layer_scores(scores, title="Distance from the middle-layer vector (placeholder metric)")

## Persist the vectors

Vectors land in `outputs/`, which is git-ignored.

In [ ]:
saved_to = engine.save_vectors(Path("../outputs/vectors/concept_vectors.pt"))
saved_to

## Next steps

- Add curated negative examples per concept so the baseline is a minimal pair
  rather than a generic neutral sentence.
- Replace the placeholder layer metric with a linear probe on held-out prompts.
- Check whether the Arabic and English examples of one concept yield the same
  direction — if they do not, the model represents them separately.
- Move to `03_concept_injection.ipynb` once a layer has been chosen.